# 05B - Bounds Stress Test Error Analysis for 1000 expanded factor (1-2, 0.25 step size) simulations



In [ ]:
import pandas as pd

name = "pediatric_dcm_patient_01_expansion_stress_test_1000_results.csv"

df = pd.read_csv("../01_Data/simulation_results/",name)

tunables = [
    "expansion_factor",
    "k_Vtot",
    "k_Vsys",
    "k_Vusv_sys",
    "k_Vusv_sys_ven",
    "k_Vusv_pulm_ven",
    "k_Ctot",
    "k_Csys",
    "k_Rsysven",
    "k_Rpulmart",
    "k_ESP_LV",
    "k_ESP_RV",
]

failed = df[df["simulation_status"] == "failed"]
success = df[df["simulation_status"] == "success"]

print("Total:", len(df))
print("Success:", len(success))
print("Failed:", len(failed))
print("Failure rate:", len(failed) / len(df) * 100)

print("\nFailed cases:")
print(failed[["simulation_id"] + tunables])

print("\nMean tunables: success vs failed")
print(df.groupby("simulation_status")[tunables].mean())

print("\nMin/max tunables in failed cases")
print(failed[tunables].describe().T)

Error analysis:

In [ ]:
from pathlib import Path
import re

ROOT = Path(__file__).resolve().parents[1]

errors_dir = (
    ROOT
    / "01_Data"
    / "Simulation_results"
    / "pediatric_dcm_patient_01"
    / "errors"
)

def normalize_error(text: str) -> str:
    """
    Remove variable parts of the error message:
    - exact simulation time
    - hmin value
    - repeated spacing / line breaks
    """

    text = text.strip()

    # Remove full MATLAB stack lines that are not useful for classification
    text = re.sub(
        r"Error in run_parallel_expansion_stress_test_pediatric_noVAD_linear.*?\n\s*\^+\n?",
        "",
        text,
        flags=re.DOTALL,
    )

    # Replace model time values
    text = re.sub(
        r"at time\s+[0-9]+(?:\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?",
        "at time <TIME>",
        text,
    )

    # Replace hmin values
    text = re.sub(
        r"hmin\s*\([0-9.]+E[+-]?[0-9]+\)",
        "hmin (<HMIN>)",
        text,
        flags=re.IGNORECASE,
    )

    # Normalize line breaks and spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def extract_main_cause(text: str) -> str:
    """
    Try to extract the meaningful cause after 'Caused by:'.
    """
    if "Caused by:" in text:
        cause = text.split("Caused by:", 1)[1]
    else:
        cause = text

    cause = normalize_error(cause)

    return cause


# ------------------------------------------------------------
# 3. Read and classify all .txt errors
# ------------------------------------------------------------

records = []

for file_path in sorted(errors_dir.glob("*.txt")):
    raw_text = file_path.read_text(encoding="utf-8", errors="ignore")

    signature = extract_main_cause(raw_text)

    records.append({
        "file": file_path.name,
        "error_signature": signature,
    })


df2 = pd.DataFrame(records)

if df2.empty:
    print("No .txt error files found.")

else:
    summary = (
        df2.groupby("error_signature")
        .agg(
            count=("file", "count"),
            example_file=("file", "first"),
        )
        .reset_index()
        .sort_values("count", ascending=False)
    )

    summary["percentage"] = (
        100 * summary["count"] / summary["count"].sum()
    ).round(2)

    display(summary)